**ASSIGNMENT WEEK-5**

GENERATE FAKE DATASET FOR DATA CLEANING PROCESS

INSTALLING LIBRARIES

In [1]:
!pip install faker pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 18.9 MB/s eta 0:00:00


IMPORT LIBRARIES

In [2]:
from faker import Faker
import pandas as pd
import random

CREATE FAKER OBJECT

In [3]:
fake = Faker()

GENERATE SAMPLE RECORDS

In [4]:
data = []

for i in range(100):

    row = {
        "user_id": random.randint(1000, 9999),
        "transaction_date": fake.date_between(
            start_date="-1y",
            end_date="today"
        ),
        "region": random.choice(
            ["North", "South", "East", "West"]
        ),
        "product_category": random.choice(
            ["Electronics", "Clothing", "Books", "Furniture"]
        ),
        "sale_amount": round(
            random.uniform(100, 5000), 2
        ),
        "city": random.choice(
            ["Delhi", "Mumbai", "Pune", "Bangalore", "Chennai"]
        ),
        "age": random.randint(18, 60),
        "subscription": random.choice(
            ["Premium", "Basic"]
        ),
        "email": fake.email(),
        "username": fake.user_name(),
        "price": round(
            random.uniform(50, 2000), 2
        ),
        "store_id": random.choice(
            ["S101", "S102", "S103", "S104"]
        ),
        "raw_timestamp": fake.date_time(),
        "status": random.choice(
            ["Active", "Inactive", "Pending"]
        )
    }

    data.append(row)

df = pd.DataFrame(data)

ADDING SOME RANDOM VALUES FOR DATA CLEANING PURPOSE

In [5]:
for col in ["email", "status", "price"]:
    df.loc[
        df.sample(frac=0.1).index,
        col
    ] = None

ADDING DUPLICATE RECORDS IN DATASET

In [6]:
duplicates = df.sample(10)

df = pd.concat(
    [df, duplicates],
    ignore_index=True
)

ADD EMPTY USERNAME

In [7]:
df.loc[
    df.sample(5).index,
    "username"
] = ""

SAVE DATASET

In [8]:
df.to_csv(
    "fake_retail_data.csv",
    index=False
)

print("Dataset Saved Successfully!")

Dataset Saved Successfully!


VERIFY DATASET

In [9]:
df.head()

df.info()

df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110 entries, 0 to 109
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   user_id           110 non-null    int64         
 1   transaction_date  110 non-null    object        
 2   region            110 non-null    object        
 3   product_category  110 non-null    object        
 4   sale_amount       110 non-null    float64       
 5   city              110 non-null    object        
 6   age               110 non-null    int64         
 7   subscription      110 non-null    object        
 8   email             99 non-null     object        
 9   username          110 non-null    object        
 10  price             100 non-null    float64       
 11  store_id          110 non-null    object        
 12  raw_timestamp     110 non-null    datetime64[ns]
 13  status            98 non-null     object        
dtypes: datetime64[ns](1), floa

,0
user_id,0
transaction_date,0
region,0
product_category,0
sale_amount,0
city,0
age,0
subscription,0
email,11
username,0


**LOADING THE DATA INTO PYSPARK**

In [10]:
!pip install pyspark

In [11]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week5Assignment") \
    .getOrCreate()

**READ CSV FILE WITH PYSPARK**

In [12]:
df = spark.read.csv(
    "fake_retail_data.csv",
    header=True,
    inferSchema=True
)

df.show(5)
df.printSchema()

+-------+----------------+------+----------------+-----------+---------+---+------------+-------------------+-------------+-------+--------+--------------------+--------+
|user_id|transaction_date|region|product_category|sale_amount|     city|age|subscription|              email|     username|  price|store_id|       raw_timestamp|  status|
+-------+----------------+------+----------------+-----------+---------+---+------------+-------------------+-------------+-------+--------+--------------------+--------+
|   3765|      2025-11-12| North|        Clothing|     660.18|     Pune| 26|       Basic|pjacobs@example.org|hugheswilliam|  975.1|    S104|1972-03-03 19:10:...| Pending|
|   7644|      2026-03-10|  West|       Furniture|    1572.95|     Pune| 19|       Basic|               NULL|    karencruz| 141.56|    S103|1995-08-26 21:08:...|  Active|
|   2798|      2026-01-01| North|     Electronics|    4365.53|Bangalore| 22|       Basic|               NULL|      terri58| 794.53|    S103|2003-

**Q3** Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date.

In [13]:
print("Rows Before:", df.count())

df_no_duplicates = df.dropDuplicates(
    ["user_id", "transaction_date"]
)

print("Rows After:", df_no_duplicates.count())

Rows Before: 110
Rows After: 100


**Q5** What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.

In [14]:
from pyspark.sql.functions import col, count, when

df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
]).show()

+-------+----------------+------+----------------+-----------+----+---+------------+-----+--------+-----+--------+-------------+------+
|user_id|transaction_date|region|product_category|sale_amount|city|age|subscription|email|username|price|store_id|raw_timestamp|status|
+-------+----------------+------+----------------+-----------+----+---+------------+-----+--------+-----+--------+-------------+------+
|      0|               0|     0|               0|          0|   0|  0|           0|   11|       5|   10|       0|            0|    12|
+-------+----------------+------+----------------+-----------+----+---+------------+-----+--------+-----+--------+-------------+------+



 **Q5** Provide a code example of filling null values in a status column with the string 'Unknown'.

In [15]:
df_filled = df.na.fill({
    "status": "Unknown"
})

df_filled.show(5)

+-------+----------------+------+----------------+-----------+---------+---+------------+-------------------+-------------+-------+--------+--------------------+--------+
|user_id|transaction_date|region|product_category|sale_amount|     city|age|subscription|              email|     username|  price|store_id|       raw_timestamp|  status|
+-------+----------------+------+----------------+-----------+---------+---+------------+-------------------+-------------+-------+--------+--------------------+--------+
|   3765|      2025-11-12| North|        Clothing|     660.18|     Pune| 26|       Basic|pjacobs@example.org|hugheswilliam|  975.1|    S104|1972-03-03 19:10:...| Pending|
|   7644|      2026-03-10|  West|       Furniture|    1572.95|     Pune| 19|       Basic|               NULL|    karencruz| 141.56|    S103|1995-08-26 21:08:...|  Active|
|   2798|      2026-01-01| North|     Electronics|    4365.53|Bangalore| 22|       Basic|               NULL|      terri58| 794.53|    S103|2003-

**Q4** Write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount.

In [16]:
from pyspark.sql.functions import avg

df.filter(
    col("region") == "West"
).groupBy(
    "product_category"
).agg(
    avg("sale_amount").alias("avg_sale")
).show()

+----------------+------------------+
|product_category|          avg_sale|
+----------------+------------------+
|     Electronics|2055.0025000000005|
|        Clothing|2491.7855555555557|
|           Books|2908.5600000000004|
|       Furniture|           1478.89|
+----------------+------------------+



**Q6** Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

In [17]:
from pyspark.sql.functions import count

city_counts = (
    df.groupBy("city")
      .agg(count("*").alias("total_records"))
)

city_counts.show()

+---------+-------------+
|     city|total_records|
+---------+-------------+
|Bangalore|           19|
|  Chennai|           13|
|   Mumbai|           24|
|     Pune|           19|
|    Delhi|           35|
+---------+-------------+



**Q8**

In [18]:
df.filter(
    (col("age") >= 18) &
    (col("age") <= 30) &
    (col("subscription") == "Premium")
).show()

+-------+----------------+------+----------------+-----------+---------+---+------------+--------------------+---------------+-------+--------+--------------------+--------+
|user_id|transaction_date|region|product_category|sale_amount|     city|age|subscription|               email|       username|  price|store_id|       raw_timestamp|  status|
+-------+----------------+------+----------------+-----------+---------+---+------------+--------------------+---------------+-------+--------+--------------------+--------+
|   4222|      2026-04-23|  West|           Books|    2506.72|     Pune| 25|     Premium|   chall@example.net|           NULL|1545.51|    S104|2015-07-13 22:09:...|  Active|
|   7494|      2025-06-11|  East|           Books|    2312.17|    Delhi| 19|     Premium|benjamincraig@exa...|     tammyewing| 481.22|    S101|1997-11-06 10:25:...|  Active|
|   3940|      2026-01-06| North|           Books|     788.36|   Mumbai| 22|     Premium| sarah08@example.net|jonathanmullins|1588

**Q12**

In [19]:
clean_df = df.filter(
    col("email").isNotNull()
).filter(
    col("username") != ""
)

clean_df.show()

+-------+----------------+------+----------------+-----------+---------+---+------------+--------------------+---------------+-------+--------+--------------------+--------+
|user_id|transaction_date|region|product_category|sale_amount|     city|age|subscription|               email|       username|  price|store_id|       raw_timestamp|  status|
+-------+----------------+------+----------------+-----------+---------+---+------------+--------------------+---------------+-------+--------+--------------------+--------+
|   3765|      2025-11-12| North|        Clothing|     660.18|     Pune| 26|       Basic| pjacobs@example.org|  hugheswilliam|  975.1|    S104|1972-03-03 19:10:...| Pending|
|   2489|      2025-10-23| North|           Books|    4344.35|   Mumbai| 21|       Basic|   jmann@example.net|        vharvey|1889.22|    S104|2005-02-22 01:52:...|  Active|
|   9114|      2025-12-25| South|           Books|    4860.95|     Pune| 28|       Basic|  mark81@example.com|      jeffrey21|1214

**Q13**

In [20]:
from pyspark.sql.functions import min, max, avg

df.agg(
    min("price").alias("min_price"),
    max("price").alias("max_price"),
    avg("price").alias("avg_price")
).show()

+---------+---------+------------------+
|min_price|max_price|         avg_price|
+---------+---------+------------------+
|    85.01|  1983.82|1075.7460000000003|
+---------+---------+------------------+



**Q15**

In [21]:
from pyspark.sql.functions import sum

final_result = (
    df
    .dropDuplicates()
    .na.fill({"price": 0})
    .groupBy("store_id")
    .agg(
        sum("price").alias("total_revenue")
    )
)

final_result.show()

+--------+------------------+
|store_id|     total_revenue|
+--------+------------------+
|    S102|          25674.29|
|    S104|30332.010000000002|
|    S101|22088.489999999998|
|    S103|          21271.58|
+--------+------------------+



**Q 10**

In [23]:
from pyspark.sql.types import TimestampType
from pyspark.sql.functions import col
df = (
    df.withColumn(
         "event_time",
         col("raw_timestamp")
         .cast(TimestampType())
         ) .drop("raw_timestamp")
    )

**Q10 VERIFICATION**

In [24]:
df.printSchema()
df.select("event_time").show(5, truncate=False)

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- event_time: timestamp (nullable = true)

+--------------------------+
|event_time                |
+--------------------------+
|1972-03-03 19:10:07.581275|
|1995-08-26 21:08:03.816477|
|2003-09-10 13:51:35.389513|
|2005-02-22 01:52:28.875878|
|2015-07-13 22:09:56.277334|
+--------------------------+
only showing top 5 rows
